In [1]:
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:

# Set the style for plots
sns.set(style="whitegrid")

# Define directories
runs_dir = "../../../../data/GOEnrichment/runs/"
plots_dir = "plots/supplement/"
os.makedirs(plots_dir, exist_ok=True)

# Find all output files in the runs/ folder (match _out.tsv and _out.sv)
file_patterns = [os.path.join(runs_dir, "*_out.tsv"), os.path.join(runs_dir, "*_out.sv")]
files = []
for pattern in file_patterns:
    files.extend(glob.glob(pattern))

print("Found files:", files)

Found files: ['../../../../data/GOEnrichment/runs\\ensembl_BP_out.tsv', '../../../../data/GOEnrichment/runs\\ensembl_CC_out.tsv', '../../../../data/GOEnrichment/runs\\ensembl_MF_out.tsv', '../../../../data/GOEnrichment/runs\\GO_BP_out.tsv', '../../../../data/GOEnrichment/runs\\GO_CC_out.tsv', '../../../../data/GOEnrichment/runs\\GO_MF_out.tsv']


In [3]:

# Iterate over each file in the runs folder
for file in files:
    try:
        df = pd.read_csv(file, sep="\t")
    except Exception as e:
        print(f"Error reading {file}: {e}")
        continue

    # Check for required columns
    required_columns = [
        "noverlap", "size", "ks_stat",
        "hg_pval", "fej_pval", "ks_pval",
        "hg_fdr", "fej_fdr", "ks_fdr"
    ]
    if not all(col in df.columns for col in required_columns):
        print(f"File {file} does not have the required columns. Skipping.")
        continue

    base_name = os.path.splitext(os.path.basename(file))[0]  # e.g., ensemb_BP_out

    # -------------------------------
    # Plot 1: Distribution of Significant Genes (SGs)
    # -------------------------------
    plt.figure(figsize=(10, 6))
    sns.histplot(df['noverlap'], bins=30, kde=True, color="darkorange")
    plt.title(f"Distribution of Significant Genes (SGs) for {base_name}")
    plt.xlabel("Number of Significant Genes (noverlap)")
    plt.ylabel("Frequency")
    plt.tight_layout()
    sg_dist_filename = os.path.join(plots_dir, f"{base_name}_sg_distribution.png")
    plt.savefig(sg_dist_filename)
    plt.close()
    print(f"Saved SG distribution plot to {sg_dist_filename}")

    # -------------------------------
    # Plot 2: Distribution of Enrichment Scores and p-values
    # (2x2 grid for KS statistic and the 3 p-values)
    # -------------------------------
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    sns.histplot(df['ks_stat'], bins=30, kde=True, ax=axes[0, 0], color="purple")
    axes[0, 0].set_title("Enrichment Score (KS Statistic)")
    axes[0, 0].set_xlabel("KS Statistic")

    sns.histplot(df['hg_pval'], bins=30, kde=True, ax=axes[0, 1], color="teal")
    axes[0, 1].set_title("Hypergeometric p-values")
    axes[0, 1].set_xlabel("hg_pval")

    sns.histplot(df['fej_pval'], bins=30, kde=True, ax=axes[1, 0], color="forestgreen")
    axes[1, 0].set_title("Fischer's Exact p-values")
    axes[1, 0].set_xlabel("fej_pval")

    sns.histplot(df['ks_pval'], bins=30, kde=True, ax=axes[1, 1], color="crimson")
    axes[1, 1].set_title("KS Test p-values")
    axes[1, 1].set_xlabel("ks_pval")

    plt.suptitle(f"Enrichment Score and p-value Distributions for {base_name}", fontsize=16)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    enr_dist_filename = os.path.join(plots_dir, f"{base_name}_enrichment_distribution.png")
    plt.savefig(enr_dist_filename)
    plt.close()
    print(f"Saved enrichment distribution plot to {enr_dist_filename}")

    # -------------------------------
    # Plot 3: Scatter Plot of KS Statistic vs. Gene Set Size
    # -------------------------------
    plt.figure(figsize=(10, 6))
    sns.scatterplot(x='size', y='ks_stat', data=df, color="navy", alpha=0.7)
    plt.title(f"KS Statistic vs. Gene Set Size for {base_name}")
    plt.xlabel("Gene Set Size")
    plt.ylabel("KS Statistic")
    plt.tight_layout()
    ks_stat_scatter_filename = os.path.join(plots_dir, f"{base_name}_ks_stat_vs_size.png")
    plt.savefig(ks_stat_scatter_filename)
    plt.close()
    print(f"Saved KS Statistic vs. size scatter plot to {ks_stat_scatter_filename}")

    # -------------------------------
    # Plot 4: Scatter Plot of KS p-value vs. Gene Set Size
    # -------------------------------
    plt.figure(figsize=(10, 6))
    sns.scatterplot(x='size', y='ks_pval', data=df, color="maroon", alpha=0.7)
    plt.title(f"KS p-value vs. Gene Set Size for {base_name}")
    plt.xlabel("Gene Set Size")
    plt.ylabel("KS p-value")
    plt.tight_layout()
    ks_pval_scatter_filename = os.path.join(plots_dir, f"{base_name}_ks_pval_vs_size.png")
    plt.savefig(ks_pval_scatter_filename)
    plt.close()
    print(f"Saved KS p-value vs. size scatter plot to {ks_pval_scatter_filename}")

    # -------------------------------
    # Plot 5: Scatter Plots of p-values vs. FDR (with y=x line)
    # Create one figure with 3 subplots for hg, Fischer's, and KS tests.
    # -------------------------------
    fig, axs = plt.subplots(1, 3, figsize=(18, 6))

    # Hypergeometric
    sns.scatterplot(x="hg_pval", y="hg_fdr", data=df, ax=axs[0], color="blue", alpha=0.7)
    # Plot y=x line
    x_limits = (df["hg_pval"].min(), df["hg_pval"].max())
    axs[0].plot(x_limits, x_limits, 'k--', label="y=x")
    axs[0].set_title("Hypergeometric: p-value vs FDR")
    axs[0].set_xlabel("hg_pval")
    axs[0].set_ylabel("hg_fdr")
    axs[0].legend()

    # Fischer's Exact
    sns.scatterplot(x="fej_pval", y="fej_fdr", data=df, ax=axs[1], color="green", alpha=0.7)
    x_limits = (df["fej_pval"].min(), df["fej_pval"].max())
    axs[1].plot(x_limits, x_limits, 'k--', label="y=x")
    axs[1].set_title("Fischer's Exact: p-value vs FDR")
    axs[1].set_xlabel("fej_pval")
    axs[1].set_ylabel("fej_fdr")
    axs[1].legend()

    # KS Test
    sns.scatterplot(x="ks_pval", y="ks_fdr", data=df, ax=axs[2], color="red", alpha=0.7)
    x_limits = (df["ks_pval"].min(), df["ks_pval"].max())
    axs[2].plot(x_limits, x_limits, 'k--', label="y=x")
    axs[2].set_title("KS Test: p-value vs FDR")
    axs[2].set_xlabel("ks_pval")
    axs[2].set_ylabel("ks_fdr")
    axs[2].legend()

    plt.suptitle(f"p-value vs FDR Scatter Plots for {base_name}", fontsize=16)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    pval_vs_fdr_filename = os.path.join(plots_dir, f"{base_name}_pval_vs_fdr.png")
    plt.savefig(pval_vs_fdr_filename)
    plt.close()
    print(f"Saved p-value vs FDR scatter plot to {pval_vs_fdr_filename}")


Saved SG distribution plot to plots/supplement/ensembl_BP_out_sg_distribution.png
Saved enrichment distribution plot to plots/supplement/ensembl_BP_out_enrichment_distribution.png
Saved KS Statistic vs. size scatter plot to plots/supplement/ensembl_BP_out_ks_stat_vs_size.png
Saved KS p-value vs. size scatter plot to plots/supplement/ensembl_BP_out_ks_pval_vs_size.png
Saved p-value vs FDR scatter plot to plots/supplement/ensembl_BP_out_pval_vs_fdr.png
Saved SG distribution plot to plots/supplement/ensembl_CC_out_sg_distribution.png
Saved enrichment distribution plot to plots/supplement/ensembl_CC_out_enrichment_distribution.png
Saved KS Statistic vs. size scatter plot to plots/supplement/ensembl_CC_out_ks_stat_vs_size.png
Saved KS p-value vs. size scatter plot to plots/supplement/ensembl_CC_out_ks_pval_vs_size.png
Saved p-value vs FDR scatter plot to plots/supplement/ensembl_CC_out_pval_vs_fdr.png
Saved SG distribution plot to plots/supplement/ensembl_MF_out_sg_distribution.png
Saved en